In [2]:
import os

print("Current Working Directory:")
print(os.getcwd())

dataset_path = "../data/processed/nyc_311_cleaned.csv"

if os.path.exists(dataset_path):
    print("\nDataset found successfully!")
else:
    print("\nDataset not found!")


Current Working Directory:
c:\Users\MY PC\Documents\5th Semester\All Projects\CivicFix\notebooks

Dataset found successfully!


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (20000, 11)


In [5]:
df.head()
print("Dataset Columns:")
for column in df.columns:
    print(column)
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())
print("Target Distribution:")
print(df['target'].value_counts())

print("\nTarget Distribution (%):")
print((df['target'].value_counts(normalize=True) * 100).round(2))

Dataset Columns:
agency
complaint_type
descriptor
incident_zip
borough
location_type
target
created_month
created_weekday
created_hour
split_date
Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   agency           20000 non-null  str  
 1   complaint_type   20000 non-null  str  
 2   descriptor       20000 non-null  str  
 3   incident_zip     20000 non-null  str  
 4   borough          20000 non-null  str  
 5   location_type    20000 non-null  str  
 6   target           20000 non-null  int64
 7   created_month    20000 non-null  int64
 8   created_weekday  20000 non-null  str  
 9   created_hour     20000 non-null  int64
 10  split_date       20000 non-null  str  
dtypes: int64(3), str(8)
memory usage: 3.4 MB

Missing Values:
agency             0
complaint_type     0
descriptor         0
incident_zip       0
borough         

In [6]:
df['split_date'] = pd.to_datetime(df['split_date'], errors='coerce')

print("Invalid dates:", df['split_date'].isnull().sum())

df = df.sort_values('split_date').reset_index(drop=True)

print("\nOldest record:", df['split_date'].iloc[0])
print("Newest record:", df['split_date'].iloc[-1])

Invalid dates: 0

Oldest record: 2020-01-03 15:41:31
Newest record: 2025-12-23 09:29:32


In [7]:
train_df = df.iloc[:16000].copy()
test_df = df.iloc[16000:].copy()

print("Training records:", len(train_df))
print("Testing records:", len(test_df))

print("\nTraining date range:")
print(train_df['split_date'].min(), "to", train_df['split_date'].max())

print("\nTesting date range:")
print(test_df['split_date'].min(), "to", test_df['split_date'].max())

Training records: 16000
Testing records: 4000

Training date range:
2020-01-03 15:41:31 to 2024-12-03 12:33:42

Testing date range:
2024-12-03 12:33:57 to 2025-12-23 09:29:32


In [8]:
feature_columns = [
    'agency',
    'complaint_type',
    'descriptor',
    'incident_zip',
    'borough',
    'location_type',
    'created_month',
    'created_weekday',
    'created_hour'
]

X_train = train_df[feature_columns].copy()
y_train = train_df['target'].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df['target'].copy()

print("Training features:", X_train.shape)
print("Training target:", y_train.shape)

print("\nTesting features:", X_test.shape)
print("Testing target:", y_test.shape)

print("\nSelected features:")
print(X_train.columns.tolist())

Training features: (16000, 9)
Training target: (16000,)

Testing features: (4000, 9)
Testing target: (4000,)

Selected features:
['agency', 'complaint_type', 'descriptor', 'incident_zip', 'borough', 'location_type', 'created_month', 'created_weekday', 'created_hour']


In [9]:
categorical_features = [
    'agency',
    'complaint_type',
    'descriptor',
    'incident_zip',
    'borough',
    'location_type',
    'created_weekday'
]

print("Unique values in categorical features:\n")

for column in categorical_features:
    print(f"{column}: {X_train[column].nunique()} unique values")

Unique values in categorical features:

agency: 16 unique values
complaint_type: 168 unique values
descriptor: 572 unique values
incident_zip: 194 unique values
borough: 7 unique values
location_type: 72 unique values
created_weekday: 7 unique values


In [10]:
numerical_features = [
    'created_month',
    'created_hour'
]

print("Numerical Feature Statistics:\n")
print(X_train[numerical_features].describe())

Numerical Feature Statistics:

       created_month  created_hour
count   16000.000000  16000.000000
mean        6.615000     13.123812
std         3.321308      6.171486
min         1.000000      0.000000
25%         4.000000      9.000000
50%         7.000000     14.000000
75%         9.000000     18.000000
max        12.000000     23.000000


In [11]:
for column in categorical_features:
    print(f"\nTop 10 values in {column}:")
    print(X_train[column].value_counts().head(10))


Top 10 values in agency:
agency
NYPD     6966
HPD      3177
DSNY     1963
DOT      1037
DEP       807
DPR       598
DOB       447
DOHMH     390
DHS       151
EDC       128
Name: count, dtype: int64

Top 10 values in complaint_type:
complaint_type
Noise - Residential                    1930
Illegal Parking                        1863
HEAT/HOT WATER                         1154
Noise - Street/Sidewalk                 847
Blocked Driveway                        728
Request Large Bulky Item Collection     642
UNSANITARY CONDITION                    485
Noise - Vehicle                         366
Street Condition                        343
PAINT/PLASTER                           278
Name: count, dtype: int64

Top 10 values in descriptor:
descriptor
Loud Music/Party                       2299
ENTIRE BUILDING                         756
Request Large Bulky Item Collection     642
Blocked Hydrant                         571
No Access                               519
Banging/Pounding         

In [12]:
import sklearn

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

print("Scikit-learn version:", sklearn.__version__)
print("Preprocessing libraries imported successfully!")

Scikit-learn version: 1.9.1
Preprocessing libraries imported successfully!


In [13]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

print("Categorical preprocessing pipeline created!")

Categorical preprocessing pipeline created!


In [14]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

print("Numerical preprocessing pipeline created!")

Numerical preprocessing pipeline created!


In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', categorical_transformer, categorical_features),
        ('numerical', numerical_transformer, numerical_features)
    ]
)

print("Complete preprocessing pipeline created!")

Complete preprocessing pipeline created!


In [16]:
X_train_processed = preprocessor.fit_transform(X_train)

print("Preprocessing completed successfully!")

print("\nOriginal training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("\nNumber of features after encoding:",
      X_train_processed.shape[1])

Preprocessing completed successfully!

Original training shape: (16000, 9)
Processed training shape: (16000, 1038)

Number of features after encoding: 1038


In [17]:
print("Processed data type:", type(X_train_processed))
print("Processed data shape:", X_train_processed.shape)

print("\nChecking for invalid numerical values:")

if hasattr(X_train_processed, "data"):
    values = X_train_processed.data
else:
    values = X_train_processed

print("Contains NaN:", np.isnan(values).any())
print("Contains Infinity:", np.isinf(values).any())

Processed data type: <class 'scipy.sparse._csr.csr_matrix'>
Processed data shape: (16000, 1038)

Checking for invalid numerical values:
Contains NaN: False
Contains Infinity: False


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

print("All three model libraries imported successfully!")

All three model libraries imported successfully!


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

print("Three models created successfully!")

Three models created successfully!


In [20]:
from sklearn.pipeline import Pipeline

logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', logistic_model)
])

logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [21]:
random_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', random_forest_model)
])

random_forest_pipeline.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [22]:
gradient_categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

gradient_numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

gradient_preprocessor = ColumnTransformer(
    transformers=[
        ('categorical',
         gradient_categorical_transformer,
         categorical_features),

        ('numerical',
         gradient_numerical_transformer,
         numerical_features)
    ]
)

print("Gradient Boosting preprocessor created successfully!")


Gradient Boosting preprocessor created successfully!


In [23]:
gradient_boosting_pipeline = Pipeline([
    ('preprocessor', gradient_preprocessor),
    ('model', gradient_boosting_model)
])

gradient_boosting_pipeline.fit(X_train, y_train)

print("Gradient Boosting trained successfully!")

Gradient Boosting trained successfully!


In [24]:
split_index = int(len(X_train) * 0.8)

X_subtrain = X_train.iloc[:split_index]
y_subtrain = y_train.iloc[:split_index]

X_val = X_train.iloc[split_index:]
y_val = y_train.iloc[split_index:]

print("Sub-training shape:", X_subtrain.shape)
print("Validation shape:", X_val.shape)

Sub-training shape: (12800, 9)
Validation shape: (3200, 9)


In [25]:
from sklearn.base import clone

logistic_validation = clone(logistic_pipeline)
random_forest_validation = clone(random_forest_pipeline)
gradient_boosting_validation = clone(gradient_boosting_pipeline)

logistic_validation.fit(X_subtrain, y_subtrain)
random_forest_validation.fit(X_subtrain, y_subtrain)
gradient_boosting_validation.fit(X_subtrain, y_subtrain)

print("All three validation models trained successfully!")

All three validation models trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    'Logistic Regression': logistic_validation,
    'Random Forest': random_forest_validation,
    'Gradient Boosting': gradient_boosting_validation
}

validation_results = []

for name, model in models.items():

    predictions = model.predict(X_val)

    validation_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_val, predictions),
        'Precision': precision_score(y_val, predictions, zero_division=0),
        'Recall': recall_score(y_val, predictions, zero_division=0),
        'F1 Score': f1_score(y_val, predictions, zero_division=0)
    })

validation_results_df = pd.DataFrame(validation_results)

print(validation_results_df.round(4).to_string(index=False))

              Model  Accuracy  Precision  Recall  F1 Score
Logistic Regression    0.7944     0.5213  0.9142    0.6639
      Random Forest    0.8291     0.5788  0.8467    0.6876
  Gradient Boosting    0.8531     0.6371  0.7876    0.7044


In [27]:
gb_configs = [
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 150, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 150, 'learning_rate': 0.05, 'max_depth': 2}
]

tuning_results = []

for i, params in enumerate(gb_configs, start=1):

    model = clone(gradient_boosting_pipeline)
    model.set_params(**{
        f'model__{key}': value
        for key, value in params.items()
    })

    model.fit(X_subtrain, y_subtrain)

    predictions = model.predict(X_val)

    tuning_results.append({
        'Configuration': i,
        'Parameters': str(params),
        'Accuracy': accuracy_score(y_val, predictions),
        'Precision': precision_score(y_val, predictions, zero_division=0),
        'Recall': recall_score(y_val, predictions, zero_division=0),
        'F1 Score': f1_score(y_val, predictions, zero_division=0)
    })

tuning_results_df = pd.DataFrame(tuning_results)

print(tuning_results_df.round(4).to_string(index=False))

 Configuration                                                   Parameters  Accuracy  Precision  Recall  F1 Score
             1  {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3}    0.8531     0.6371  0.7876    0.7044
             2  {'n_estimators': 150, 'learning_rate': 0.1, 'max_depth': 3}    0.8556     0.6394  0.8031    0.7120
             3 {'n_estimators': 150, 'learning_rate': 0.05, 'max_depth': 2}    0.8453     0.6301  0.7356    0.6788


In [28]:
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gradient_boosting_pipeline = Pipeline([
    ('preprocessor', gradient_preprocessor),
    ('model', gradient_boosting_model)
])

print("Best Gradient Boosting configuration selected!")

Best Gradient Boosting configuration selected!


In [29]:
logistic_pipeline.fit(X_train, y_train)

random_forest_pipeline.fit(X_train, y_train)

gradient_boosting_pipeline.fit(X_train, y_train)

print("All final models trained successfully on 16,000 records!")

All final models trained successfully on 16,000 records!


In [30]:
import os
import joblib

os.makedirs('../models', exist_ok=True)

joblib.dump(logistic_pipeline, '../models/logistic_regression.pkl')

joblib.dump(random_forest_pipeline, '../models/random_forest.pkl')

joblib.dump(gradient_boosting_pipeline, '../models/gradient_boosting.pkl')

print("All three models saved successfully!")

print("\nSaved files:")
print(os.listdir('../models'))

All three models saved successfully!

Saved files:
['gradient_boosting.pkl', 'logistic_regression.pkl', 'random_forest.pkl']


In [31]:
import joblib

# Load saved models
loaded_logistic = joblib.load('../models/logistic_regression.pkl')
loaded_random_forest = joblib.load('../models/random_forest.pkl')
loaded_gradient_boosting = joblib.load('../models/gradient_boosting.pkl')

sample_complaint = X_train.iloc[[0]]

print("Logistic Regression:", loaded_logistic.predict(sample_complaint)[0])
print("Random Forest:", loaded_random_forest.predict(sample_complaint)[0])
print("Gradient Boosting:", loaded_gradient_boosting.predict(sample_complaint)[0])

print("\nAll saved models loaded and tested successfully!")

Logistic Regression: 1
Random Forest: 1
Gradient Boosting: 0

All saved models loaded and tested successfully!
